# Offline Evaluation Assignment — LangSmith

In this assignment you will run a complete **offline evaluation** workflow on LangSmith:

1. **Create a dataset** locally and export it as a CSV.
2. **Download the CSV** and upload it to LangSmith through the UI to create a dataset.
3. **Connect** back to that dataset from this notebook using its name and **extract** the examples.
4. **Define a target model** whose outputs you want to evaluate.
5. **Run two LLM-based evaluators** (Correctness and Conciseness) and **log an experiment** to LangSmith.

> **Offline evaluation** means you evaluate against a *fixed, curated dataset* with known reference answers — as opposed to *online* evaluation, which scores live production traces.

Work through the notebook **top to bottom**, one section at a time.


---
# Section 0 — Setup

Install dependencies, import libraries, and configure your Azure OpenAI and LangSmith credentials.


### 0.1 — Install dependencies

In [ ]:
!pip install -q langsmith openai pandas

### 0.2 — Imports

In [ ]:
import os
import json
import re

import pandas as pd
from openai import AzureOpenAI
from langsmith import Client
from langsmith.wrappers import wrap_openai

### 0.3 — Configure Azure OpenAI

Fill in your Azure OpenAI details below — **endpoint**, **API version**, **deployment name**, and **API key**.

> ⚠️ Replace the `YOUR_..._HERE` placeholders with your own values. Avoid sharing this notebook (or a Colab share link) while real keys are pasted in, since anyone with the link could use them.


In [ ]:
os.environ["OPENAI_API_TYPE"]    = "azure"
os.environ["OPENAI_API_BASE"]    = "htopenai.azure.com/"   # <-- your endpoint
os.environ["OPENAI_API_VERSION"] = "2024eview"
os.environ["OPENAI_API_KEY"]     = "CLGZNOw3AAABACOGHS5v"           # <-- your key
os.environ["OPENAI_DEPLOYMENT"]  = "gpt-5"                                     # <-- your deployment name

azure = wrap_openai(
    AzureOpenAI(
        api_key=os.getenv("OPENAI_API_KEY"),
        azure_endpoint=os.getenv("OPENAI_API_BASE"),
        api_version=os.getenv("OPENAI_API_VERSION"),
    )
)

DEPLOYMENT = os.getenv("OPENAI_DEPLOYMENT")
print("Azure OpenAI client ready. Deployment:", DEPLOYMENT)

Azure OpenAI client ready. Deployment: gpt-5


**Quick connection test** (optional but recommended):

> Note: `gpt-5` and o-series reasoning deployments require `max_completion_tokens` (not `max_tokens`) and only accept the default `temperature`. The calls in this notebook follow that convention so they work across model types.


In [ ]:
try:
    resp = azure.chat.completions.create(
        model=DEPLOYMENT,
        messages=[{"role": "user", "content": "Reply with the single word: ready"}],
        max_completion_tokens=2000,
    )
    print("Connection OK ->", resp.choices[0].message.content.strip())
except Exception as e:
    print("Connection failed:", e)

Connection OK -> ready


### 0.4 — Configure LangSmith

Paste your LangSmith API key below.


In [ ]:
os.environ["LANGSMITH_API_KEY"] = "lsvb353a5a7"   # <-- your key

client = Client(api_key=os.environ["LANGSMITH_API_KEY"])
print("LangSmith client initialized.")

LangSmith client initialized.


---
# Section 1 — Create the Evaluation Dataset (CSV)

We build a small question-answer dataset in memory, export it to a CSV file, and download it. Each row has:

- **`question`** — the input that will be sent to the model.
- **`reference_answer`** — the gold / expected answer used by the evaluators.


### 1.1 — Define the examples

Feel free to edit, add, or replace rows. Keep the two column names (`question`, `reference_answer`) so the upload mapping in Section 1.3 stays simple.


In [ ]:
import pandas as pd

# Load the Excel dataset
file_path = '/content/PolicyBot_Sample_QA.xlsx'
df_raw = pd.read_excel(file_path)

# Map columns including the pre-existing model output
examples_df = df_raw.rename(columns={
    "Input": "question",
    "Expected_Output": "reference_answer",
    "Expected_Context": "context",
    "Actual_Output": "actual_answer"
})

# Convert to list of dicts for SDK upload
examples = examples_df.to_dict(orient="records")

print(f"Loaded {len(examples)} examples with pre-computed answers.")
display(examples_df.head())

Loaded 20 examples with pre-computed answers.


,question,reference_answer,context,actual_answer,Actual_Context
0,How many earned leaves can an employee carry f...,Up to 30 days of EL can be carried forward; ba...,"As per the leave policy, an employee may carry...",Employees can carry forward only up to 25 days...,"As per the leave policy, unused Earned Leave m..."
1,What is the per diem allowance for internation...,USD 75/day for USA/UK/Western Europe/Singapore...,The travel policy prescribes a per diem of USD...,The per diem allowance is USD 75 per day for t...,International per diem: USD 75 per day for tra...
2,How many dependents are covered under medical ...,Maximum 2 dependents under the base GMC policy...,"Under the base Group Medical Cover, an employe...",A maximum of 4 dependents can be covered under...,An employee may enroll a maximum of 4 dependen...
3,"Who approves expense claims above Rs. 50,000?",The relevant Vice President approves claims ab...,"Expense claims exceeding Rs. 50,000 require ap...","Expense claims above Rs. 50,000 are approved b...","Expense claims above Rs. 50,000 require approv..."
4,Can an employee on probation avail WFH?,Probationers are NOT eligible for standard WFH...,Employees serving their probation period are n...,"No, probationers are not eligible for standard...",All confirmed employees of Meridian Technologi...


### 1.2 — Choose your upload method:

You have two options to get your dataset into LangSmith:



**Option A: Direct SDK Upload (Run the cell below)**
Upload the dataset directly using the LangSmith Python SDK. If you choose this option, you can skip Section 1.3 (UI upload instructions) and proceed to Section 2 to extract examples using the same `DATASET_NAME`.

**Option B: Manual CSV Upload (Continue below)**
Download the `offline_eval_dataset.csv` file, then manually upload it to LangSmith through the UI (as described in Section 1.3).

### Option A - Direct SDK Upload

In [ ]:
# Option A: Direct SDK Upload
from langsmith.schemas import DataType
DATASET_NAME = "policy-bot-live-v2"

# Create the dataset
try:
    dataset = client.create_dataset(
        dataset_name=DATASET_NAME,
        description="PolicyBot evaluation for live generation.",
        data_type=DataType.kv,
    )
except Exception:
    dataset = client.read_dataset(dataset_name=DATASET_NAME)

# Upload examples - we send the 'Expected_Context' as an input called 'expected_context'
for ex in examples:
    client.create_example(
        dataset_id=dataset.id,
        inputs={
            "question": ex["question"],
            "expected_context": ex["context"]
        },
        outputs={"reference_answer": ex["reference_answer"]},
    )

print(f"Uploaded examples to dataset '{DATASET_NAME}'.")

Uploaded examples to dataset 'policy-bot-live-v2'.


### Option B — Save the CSV and download it

In [ ]:
CSV_PATH = "offline_eval_dataset.csv"
# Use the correct variable name 'examples_df' from section 1.1
examples_df.to_csv(CSV_PATH, index=False)
print("Saved:", CSV_PATH)

# Trigger a browser download in Colab
from google.colab import files
files.download(CSV_PATH)

Saved: offline_eval_dataset.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### 1.3 — Upload the CSV to LangSmith (UI steps)

Now switch to the **LangSmith web app** and create the dataset from your downloaded CSV:

1. Go to **https://smith.langchain.com** and open **Datasets & Experiments**.
2. Click **+ New Dataset** → choose **Upload CSV** (or **Import from CSV**).
3. Select the `offline_eval_dataset.csv` file you just downloaded.
4. Give the dataset a clear **name** — you'll need it in the next section (e.g. `offline-eval-qa`).
5. **Map the columns:**
   - `question` → **Input** key
   - `reference_answer` → **Output** key (reference)
6. Confirm and create the dataset.

> 📝 **Remember the exact dataset name** — Section 2 needs it verbatim.


---
# Section 2 — Connect to the Uploaded Dataset

Point this notebook at the dataset you just created in the UI, then pull the examples back down to confirm everything loaded correctly.


### 2.1 — Enter the dataset name

Set this to the **exact** name you gave the dataset in LangSmith.


In [ ]:
DATASET_NAME = "policy-bot-live-v2"   # <-- change to your dataset name to the one which you have set

### 2.2 — Extract the examples

Fetch the examples from LangSmith and preview them. This both verifies the connection and shows the **input/output key names** the evaluators must use.


In [ ]:
examples = list(client.list_examples(dataset_name=DATASET_NAME))
print(f"Fetched {len(examples)} examples from dataset '{DATASET_NAME}'.\n")

for i, ex in enumerate(examples[:5], start=1):
    print(f"--- Example {i} ---")
    print("inputs :", dict(ex.inputs))
    print("outputs:", dict(ex.outputs))
    print()

if examples:
    print("Input keys :", list(examples[0].inputs.keys()))
    print("Output keys:", list(examples[0].outputs.keys()))

Fetched 40 examples from dataset 'policy-bot-live-v2'.

--- Example 1 ---
inputs : {'question': 'What rating distribution is enforced during calibration?', 'expected_context': 'During calibration, the rating distribution is enforced such that no more than 15% of employees may be rated Outstanding or Exceeds combined, and no more than 10% rated Below or Partially Meets combined.'}
outputs: {'reference_answer': 'Not more than 15% combined Outstanding+Exceeds; not more than 10% combined Below+Partially Meets.'}

--- Example 2 ---
inputs : {'question': 'How many quotations are required for procurement above Rs. 1,00,000?', 'expected_context': 'For any procurement exceeding Rs. 1,00,000, a minimum of 3 written quotations must be obtained from empanelled vendors.'}
outputs: {'reference_answer': 'Minimum of 3 written quotations from empanelled vendors.'}

--- Example 3 ---
inputs : {'question': 'What is the maximum continuous earned leave that can be taken at a stretch?', 'expected_context': 

> If the **input/output key names** printed above are not `question` and `reference_answer` (for example, if the UI capitalised them), update the keys used in the target function and evaluators in the next sections to match.


---
# Section 3 — Define the App under Test

This is the system **being evaluated**. For each example, LangSmith passes the example's `inputs` dict to this function; it returns a dict of outputs that the evaluators will score.


In [ ]:
def target(inputs: dict) -> dict:
    """
    App under test: A sophisticated RAG simulator that generates
    grounded answers and simulated retrieved context.
    """
    question = inputs.get("question", "")
    expected_context = inputs.get("expected_context", "")

    prompt = f"""### TASK:
You are a highly accurate Policy Assistant RAG system. Your goal is to formulate a professional and clear answer based strictly on the provided context.

### CONSTRAINTS:
- Use ONLY the provided 'Expected Context'. Do not use outside knowledge.
- If the context does not contain the answer, explicitly state that.
- Provide an 'actual_context' field representing the specific excerpt used.
- Return the response in strict JSON format.

### DATA:
- Expected Context: {expected_context}
- Question: {question}

### OUTPUT FORMAT:
{{
  "answer": "your grounded answer here",
  "context": "the specific excerpt used"
}}"""

    response = azure.chat.completions.create(
        model=DEPLOYMENT,
        messages=[{"role": "system", "content": "You are a helpful RAG assistant."}, {"role": "user", "content": prompt}],
        max_completion_tokens=2000,
        response_format={ "type": "json_object" }
    )

    res_json = json.loads(response.choices[0].message.content)
    return {
        "answer": res_json.get("answer", ""),
        "context": res_json.get("context", "")
    }

---
# Section 4 — Define the LLM-Based Evaluators

We define **two** LLM-as-judge evaluators. Each receives:

- `inputs` — the example inputs (e.g. `question`)
- `outputs` — what the **target model** returned (e.g. `answer`)
- `reference_outputs` — the gold answer from the dataset (e.g. `reference_answer`)

and returns a dict with a `key`, a numeric `score`, and a short `comment`.

A small helper parses the judge's JSON reply safely.


In [ ]:
def _parse_judge_json(text: str) -> dict:
    """Parse a JSON object from an LLM reply, tolerating extra prose / code fences."""
    try:
        return json.loads(text)
    except Exception:
        m = re.search(r"\{.*\}", text, flags=re.DOTALL)
        if m:
            try:
                return json.loads(m.group(0))
            except Exception:
                pass
    return {}

### 4.1 — Evaluator 1: Correctness

Evaluates whether the model's answer is **factually correct and complete with respect to the reference answer**.

The evaluator compares the generated answer against the reference answer and assesses whether:
- All required information is present.
- The answer contains no factual inaccuracies.
- Every part of the question has been addressed.
- The response is logically consistent.
- Appropriate and accurate terminology is used.

The evaluator assigns a **binary score**:
- **1** — The answer is factually correct, complete, and consistent with the reference.
- **0** — The answer contains factual errors, omissions, incorrect terminology, logical inconsistencies, or fails to answer the question completely.

Along with the score, the evaluator generates a detailed explanation describing how the generated answer compares to the reference and highlighting any strengths, omissions, or inaccuracies.

In [ ]:
def correctness(inputs: dict, outputs: dict, reference_outputs: dict) -> dict:
    question  = inputs.get("question", "")
    answer    = outputs.get("answer", "")
    reference = reference_outputs.get("reference_answer", "")

    prompt = f"""You are an expert data labeler evaluating model outputs for correctness. Your task is to assign a score based on the following rubric:

<Rubric>
  A correct answer:
  - Provides accurate and complete information
  - Contains no factual errors
  - Addresses all parts of the question
  - Is logically consistent
  - Uses precise and accurate terminology

  When scoring, you should penalize:
  - Factual errors or inaccuracies
  - Incomplete or partial answers
  - Misleading or ambiguous statements
  - Incorrect terminology
  - Logical inconsistencies
  - Missing key information
</Rubric>

<Instructions>
  - Carefully read the input and output.
  - Check for factual accuracy and completeness.
  - Return the results in a valid JSON format with the keys: "score" (int 0 or 1) and "explanation" (string).
  - The "explanation" MUST be a detailed, multi-line analysis. Compare the model's answer point-by-point against the reference, identifying specific strengths, omissions, or errors.
</Instructions>

<input>
{inputs}
</input>

<output>
{outputs}
</output>

<reference_outputs>
{reference_outputs}
</reference_outputs>"""

    resp = azure.chat.completions.create(
        model=DEPLOYMENT,
        messages=[{"role": "user", "content": prompt}],
        max_completion_tokens=2000,
        response_format={ "type": "json_object" }
    )

    parsed = _parse_judge_json(resp.choices[0].message.content)

    return {
        "key": "correctness",
        "score": int(parsed.get("score", 0)),
        "comment": parsed.get("explanation", "No explanation provided."),
    }

### 4.2 — Evaluator 2: Conciseness

Evaluates whether the generated answer is **concise while still providing the information requested**, independent of factual correctness.

The evaluator examines the answer for unnecessary verbosity and rewards responses that:
- Contain only the information needed to answer the question.
- Use the minimum number of words necessary.
- Avoid redundant statements, filler, pleasantries, and hedging language.
- Exclude unnecessary explanations unless the question explicitly requests them.

The evaluator assigns a **continuous score between 0 and 1**:
- **1.0** — Perfectly concise; contains only the essential information.
- **0.75** — Mostly concise with minor unnecessary wording.
- **0.50** — Noticeably verbose but still focused on the requested information.
- **0.25** — Contains significant unnecessary wording or explanations.
- **0.0** — Extremely verbose or largely composed of unnecessary content.

The evaluator measures **only conciseness**. It intentionally ignores factual correctness, completeness, faithfulness, relevance, and writing quality, as those are evaluated by separate metrics. Along with the score, it returns a brief explanation describing why the assigned score was given.

In [ ]:
def conciseness(inputs: dict, outputs: dict, reference_outputs: dict) -> dict:
    question = inputs.get("question", "")
    answer = outputs.get("answer", "")

    prompt = f"""You are an expert data labeler evaluating model outputs for conciseness. Your task is to assign a score based only on how concise the answer is.

<Rubric>
A perfectly concise answer:
- Contains only the exact information requested.
- Uses the minimum number of words necessary to convey the complete answer.
- Omits pleasantries, hedging language, and unnecessary context.
- Excludes meta-commentary about the answer or the model's capabilities.
- Avoids redundant information or restatements.
- Does not include explanations unless explicitly requested.

When scoring, you should deduct points for:
- Introductory phrases like "I believe," "I think," or "The answer is."
- Hedging language like "probably," "likely," or "as far as I know."
- Unnecessary context or background information.
- Explanations when the question does not ask for them.
- Follow-up questions or offers for more information.
- Redundant information or restatements.
- Polite phrases like "Hope this helps" or "Let me know if you need anything else."
</Rubric>

<Scoring>
Assign a score between 0 and 1.

- 1.0 = Perfectly concise. Contains only the essential information requested.
- 0.75 = Mostly concise with only minor unnecessary wording.
- 0.50 = Noticeably verbose but still focused on the requested information.
- 0.25 = Contains substantial unnecessary wording or explanations.
- 0.0 = Extremely verbose or largely composed of unnecessary content.
</Scoring>

<Instructions>
- Read the question and answer carefully.
- Evaluate ONLY conciseness.
- Do NOT evaluate factual correctness, completeness, faithfulness, relevance, or writing quality. Those are separate evaluation dimensions.
- Consider whether explanations are appropriate based on what the question asks.
- The score should reflect how close the response comes to containing only the essential information requested.
- Return ONLY a valid JSON object with the following format:

{{
  "score": <number between 0 and 1>,
  "explanation": "<brief explanation for the assigned score>"
}}
</Instructions>

<Reminder>
The goal is to reward responses that provide complete answers with absolutely no extraneous information.
</Reminder>

<Question>
{question}
</Question>

<Answer>
{answer}
</Answer>
"""

    resp = azure.chat.completions.create(
        model=DEPLOYMENT,
        messages=[{"role": "user", "content": prompt}],
        max_completion_tokens=2000,
        response_format={"type": "json_object"},
    )

    parsed = _parse_judge_json(resp.choices[0].message.content)

    return {
        "key": "conciseness",
        "score": float(parsed.get("score", 0.0)),
        "comment": parsed.get("explanation", ""),
    }

### 4.3 — Evaluator 3: Faithfulness

Evaluates whether the generated answer is **fully grounded in the retrieved context** without introducing unsupported information or hallucinations.

The evaluator decomposes the generated answer into individual factual claims and verifies whether each claim is supported by the retrieved context. Claims may be supported either explicitly or through direct logical inference, and semantic equivalence or paraphrasing is accepted.

The evaluator assigns a **continuous score between 0 and 1**, calculated as:

> **Faithfulness = Supported Claims / Total Claims**

A score of:
- **1.0** indicates that every factual claim in the answer is supported by the retrieved context.
- **0.0** indicates that none of the claims are supported.
- Intermediate scores represent the proportion of supported claims.

The evaluator penalizes:
- Hallucinated or unsupported claims.
- Statements that contradict the retrieved context.
- Over-generalizations or use of external knowledge not present in the retrieved context.

The evaluator intentionally **does not** assess completeness, conciseness, relevance, or writing quality, as those are evaluated by separate metrics. Along with the score, it returns a brief explanation describing which claims were or were not supported by the retrieved context.

In [ ]:
def faithfulness(inputs: dict, outputs: dict, reference_outputs: dict) -> dict:
    actual_context = outputs.get("context", "")
    answer = outputs.get("answer", "")

    prompt = f"""You are an expert data labeler evaluating model outputs for faithfulness.

Your task is to determine whether the answer is fully supported by the provided context.

<Rubric>

A faithful answer:
- Is derived solely from the provided context.
- Does not introduce outside information or hallucinations.
- Is factually consistent with the retrieved context.
- Can be directly supported or logically inferred from the provided context.
- May use different wording or paraphrasing while preserving the same meaning.

When scoring, you should penalize:
- Claims that contradict the context.
- Claims that are not supported by the context.
- Hallucinated facts.
- Over-generalizations that extend beyond the provided context.
- Use of external knowledge not present in the context.

Do NOT penalize:
- Missing information.
- Lack of completeness.
- Conciseness or verbosity.
- Writing style.
- Answer relevance.

These are evaluated by separate metrics.

</Rubric>

<Scoring>

1. Break the answer into independent factual claims.

2. For each claim, determine whether it is fully supported by the provided context.

3. Compute the score as:

supported_claims / total_claims

Examples:
- 5 of 5 supported → 1.0
- 4 of 5 supported → 0.8
- 2 of 4 supported → 0.5
- 0 of 3 supported → 0.0

</Scoring>

<Instructions>

- Analyze the answer claim by claim.
- Verify each claim using only the provided context.
- Accept semantic equivalence and paraphrasing.
- Do not use outside knowledge.
- Return ONLY a valid JSON object in the following format:

{{
  "score": <number between 0 and 1>,
  "explanation": "<brief explanation of why the score was assigned>"
}}

</Instructions>

<Reminder>

The goal is to ensure the model remains completely grounded in the retrieved context and does not hallucinate unsupported information.

</Reminder>

<Context>
{actual_context}
</Context>

<Answer>
{answer}
</Answer>
"""

    resp = azure.chat.completions.create(
        model=DEPLOYMENT,
        messages=[{"role": "user", "content": prompt}],
        max_completion_tokens=2000,
        response_format={"type": "json_object"},
    )

    parsed = _parse_judge_json(resp.choices[0].message.content)

    return {
        "key": "faithfulness",
        "score": float(parsed.get("score", 0.0)),
        "comment": parsed.get("explanation", ""),
    }

### 4.4 — Evaluator 4: Contextual Recall

Evaluates whether the **retrieved context contains all the information required to produce the reference answer**. This metric measures the effectiveness of the retrieval component rather than the quality of the generated answer.

The evaluator decomposes the reference answer into individual factual claims and verifies whether each claim is supported by the retrieved context. Claims may be supported either explicitly or through direct logical inference, and semantic equivalence or paraphrasing is accepted.

The evaluator assigns a **continuous score between 0 and 1**, calculated as:

> **Contextual Recall = Supported Claims / Total Claims**

A score of:
- **1.0** indicates that the retrieved context contains all the information needed to produce the reference answer.
- **0.0** indicates that none of the required information is present.
- Intermediate scores represent the proportion of reference claims supported by the retrieved context.

The evaluator penalizes:
- Missing factual information required by the reference answer.
- Missing numerical values, dates, names, or conditions.
- Missing constraints or requirements.
- Retrieved context that is related to the topic but does not contain the specific information needed to support the reference answer.

Unlike **Faithfulness**, which evaluates whether the generated answer is grounded in the retrieved context, **Contextual Recall** evaluates whether the retrieved context itself is sufficiently complete to support the expected answer. Along with the score, the evaluator returns a brief explanation summarizing which reference claims were unsupported by the retrieved context.

In [ ]:
def contextual_recall(inputs: dict, outputs: dict, reference_outputs: dict) -> dict:
    actual_context = outputs.get("context", "")
    reference = reference_outputs.get("reference_answer", "")

    prompt = f"""You are an expert evaluator assessing the Contextual Recall of a retrieval system.

Your goal is to determine whether the retrieved context contains all of the information required to produce the reference answer. Return the results in JSON format.

<Rubric>

1. Break the reference answer into its independent factual claims.
   - Each standalone fact should be treated as a separate claim.
   - Numerical values, dates, names, conditions, requirements, and constraints should each be considered when determining whether a claim is fully supported.

2. For each claim, determine whether it is fully supported by the retrieved context.
   - A claim is supported if the information is explicitly present or can be directly inferred from the retrieved context.
   - Minor wording differences or paraphrases are acceptable.
   - Do NOT require exact string matches.

3. A claim is NOT supported if:
   - Any important detail is missing.
   - Numerical values differ or are absent.
   - Dates are missing.
   - Required conditions or constraints are missing.
   - The context is only generally related but does not contain the specific fact.

4. Compute the Contextual Recall score as:

        supported_claims / total_claims

Examples:
- 5 of 5 supported → 1.0
- 4 of 5 supported → 0.8
- 2 of 4 supported → 0.5
- 0 of 3 supported → 0.0

</Rubric>

<Instructions>

Analyze the reference answer and retrieved context.

First:
- Extract the factual claims from the reference answer.

Then:
- Determine whether each claim is supported by the retrieved context.

Finally return ONLY a JSON object with the following schema:

{{
  "claims": [
    {{
      "claim": "...",
      "supported": true,
      "reason": "..."
    }}
  ],
  "score": <float between 0 and 1>,
  "explanation": "Brief explanation summarizing which claims were unsupported."
}}

</Instructions>

<RetrievedContext>
{actual_context}
</RetrievedContext>

<ReferenceAnswer>
{reference}
</ReferenceAnswer>
"""

    resp = azure.chat.completions.create(
        model=DEPLOYMENT,
        messages=[{"role": "user", "content": prompt}],
        max_completion_tokens=2000,
        response_format={ "type": "json_object" }
    )
    parsed = _parse_judge_json(resp.choices[0].message.content)

    return {
        "key": "contextual_recall",
        "score": float(parsed.get("score", 0.0)),
        "comment": parsed.get("explanation", ""),
    }

### 4.5 — Running the Evaluation

Once the target application and custom evaluators have been defined, the evaluation is executed using the LangSmith `client.evaluate()` API.

#### Parameters

- **`target`**  
  The application or callable under evaluation. For each example in the dataset, LangSmith invokes this function to generate the model's response.

- **`data=DATASET_NAME`**  
  Specifies the evaluation dataset stored in LangSmith. Each dataset example contains the input question and the corresponding reference answer used by the evaluators.

- **`evaluators=[...]`**  
  A list of custom evaluators executed for every dataset example. In this implementation, four evaluators are applied:
  - **Correctness** – Compares the generated answer against the reference answer.
  - **Conciseness** – Measures whether the answer contains unnecessary information.
  - **Faithfulness** – Verifies that the generated answer is fully supported by the retrieved context.
  - **Contextual Recall** – Evaluates whether the retrieved context contains all information required to produce the reference answer.

- **`experiment_prefix="policy-bot-live-rag"`**  
  Specifies the prefix used when creating the LangSmith experiment. LangSmith automatically appends a unique identifier, allowing multiple evaluation runs to be grouped while remaining distinguishable.

- **`max_concurrency=2`**  
  Controls the number of evaluation examples processed concurrently. Increasing this value can reduce overall evaluation time, subject to API rate limits and available compute resources.

#### Evaluation Workflow

For each example in the dataset, LangSmith performs the following steps:

1. Executes the target application using the dataset input.
2. Collects the generated answer and retrieved context.
3. Runs each custom evaluator independently.
4. Records the evaluator score and explanation as feedback.
5. Stores all results within a LangSmith experiment for later inspection and analysis.

The resulting experiment contains:
- The input question.
- The generated answer.
- The retrieved context.
- Individual evaluator scores.
- Detailed evaluator reasoning.
- Execution traces for debugging and analysis.

The `ExperimentResults` object returned by `client.evaluate()` provides a handle to the completed evaluation run, while the full results, traces, and feedback are available through the LangSmith UI.

In [ ]:
experiment_results = client.evaluate(
    target,
    data=DATASET_NAME,
    evaluators=[correctness, conciseness, faithfulness, contextual_recall],
    experiment_prefix="policy-bot-live-rag",
    max_concurrency=2,
)

print("Evaluation complete.")
print(experiment_results)

View the evaluation results for experiment: 'policy-bot-live-rag-5a33e06a' at:
https://smith.langchain.com/o/2633fe0a-9587-4e9b-9ff1-685d7aaba413/datasets/2db03d05-cfe8-47ba-9958-6e7a20777f67/compare?selectedSessions=12fc8441-985b-443a-bd7a-dd2db3920c27




0it [00:00, ?it/s]

Evaluation complete.
<ExperimentResults policy-bot-live-rag-5a33e06a>


## 5 — Pairwise Evaluation

Pairwise evaluation is used to **directly compare the outputs of two different experiments** on the same dataset. Instead of assigning an absolute score to a single response, the evaluator determines **which of the two responses is better** for each dataset example.

This approach is particularly useful for comparing:
- Different prompt versions.
- Different LLMs.
- Different retrieval strategies.
- Different application versions.

### Workflow

The pairwise evaluation process consists of the following steps:

1. Execute two experiments using the same LangSmith dataset.
2. For each dataset example, retrieve the outputs from both experiments.
3. Pass both outputs to a custom pairwise evaluator.
4. Compare the responses according to predefined evaluation criteria.
5. Select the preferred response (Experiment A, Experiment B, or Tie).
6. Store the comparison results as a Pairwise Experiment in LangSmith.

### Pairwise Evaluator

Unlike standard evaluators, which score a single response independently, a pairwise evaluator receives the outputs from **two experiment runs** for the same dataset example.

The evaluator compares both responses simultaneously and returns:
- The preferred response (**A**, **B**, or **Tie**).
- A reasoning explaining the decision.

For example, a pairwise evaluator may compare responses based on:
- Correctness
- Completeness
- Clarity
- Conciseness
- Faithfulness
- Overall response quality

### Running the Pairwise Experiment

Once both experiments have completed, the pairwise evaluation can be executed using LangSmith's comparative evaluation API.

```python
from langsmith.evaluation import evaluate_comparative

pairwise_results = evaluate_comparative(
    experiments=(
        experiment_a.experiment_name,
        experiment_b.experiment_name,
    ),
    evaluators=[
        pairwise_preference,
    ],
    experiment_prefix="policy-bot-pairwise",
)

print(pairwise_results)
```

### Results

LangSmith creates a dedicated **Pairwise Experiment** containing the comparison results for every dataset example.

For each example, the experiment records:
- The output generated by Experiment A.
- The output generated by Experiment B.
- The preferred response (A, B, or Tie).
- The evaluator's reasoning.

LangSmith also provides aggregate statistics summarizing how frequently each experiment was preferred, enabling straightforward comparison of prompt versions, model versions, or application revisions.

In [ ]:
from langsmith.evaluation import evaluate_comparative
from langsmith.schemas import Run, Example

## 5.1 — Run Experiment A

The first experiment evaluates the baseline version of the application. Its outputs will later be compared against another experiment using pairwise evaluation.

In [ ]:
experiment_a = client.evaluate(
    target,
    data=DATASET_NAME,
    evaluators=[
        correctness,
        conciseness,
        faithfulness,
        contextual_recall,
    ],
    experiment_prefix="policy-bot-v1",
    max_concurrency=2,
)

print(experiment_a)

View the evaluation results for experiment: 'policy-bot-v1-8d570ef2' at:
https://smith.langchain.com/o/2633fe0a-9587-4e9b-9ff1-685d7aaba413/datasets/2db03d05-cfe8-47ba-9958-6e7a20777f67/compare?selectedSessions=290130c2-04de-4a3a-86dd-a11f0cba4382




0it [00:00, ?it/s]

<ExperimentResults policy-bot-v1-8d570ef2>


## 5.2 — Run Experiment B

The second experiment evaluates another version of the application (for example, a different prompt, model, or retrieval strategy). It is executed on the same dataset so that both experiments produce outputs for identical examples.

In [ ]:
experiment_b = client.evaluate(
    target,
    data=DATASET_NAME,
    evaluators=[
        correctness,
        conciseness,
        faithfulness,
        contextual_recall,
    ],
    experiment_prefix="policy-bot-v2",
    max_concurrency=2,
)

print(experiment_b)

View the evaluation results for experiment: 'policy-bot-v2-f63ddcd9' at:
https://smith.langchain.com/o/2633fe0a-9587-4e9b-9ff1-685d7aaba413/datasets/2db03d05-cfe8-47ba-9958-6e7a20777f67/compare?selectedSessions=ff345445-183b-47aa-b1ea-19298d03b8ff




0it [00:00, ?it/s]

<ExperimentResults policy-bot-v2-f63ddcd9>


## 5.3 — Pairwise Evaluator

A pairwise evaluator receives the outputs from two experiment runs corresponding to the same dataset example.

The evaluator compares both responses using an LLM judge and returns:

- The preferred response (**A**, **B**, or **Tie**)
- A reasoning explaining the decision

Unlike single-response evaluators, the decision is made by considering **both responses simultaneously**.

In [ ]:
def pairwise_preference(runs: list[Run], example: Example):
    run_a, run_b = runs

    question = example.inputs["question"]
    reference = example.outputs["reference_answer"]

    answer_a = run_a.outputs.get("answer", "")
    answer_b = run_b.outputs.get("answer", "")

    prompt = f"""
You are an expert evaluator.

Compare two answers to the same question.

<Question>
{question}
</Question>

<Reference Answer>
{reference}
</Reference Answer>

<Answer A>
{answer_a}
</Answer A>

<Answer B>
{answer_b}
</Answer B>

Evaluate both answers based on:

- Correctness
- Completeness
- Clarity
- Precision

Return ONLY a JSON object:

{{
    "preferred": "A" | "B" | "Tie",
    "reasoning": "<brief explanation>"
}}
"""

    response = azure.chat.completions.create(
        model=DEPLOYMENT,
        messages=[
            {
                "role": "user",
                "content": prompt,
            }
        ],
        response_format={"type": "json_object"},
    )

    parsed = _parse_judge_json(
        response.choices[0].message.content
    )

    return {
        "key": "pairwise_preference",
        "scores": {
            run_a.id: 1 if parsed["preferred"] == "A" else 0,
            run_b.id: 1 if parsed["preferred"] == "B" else 0,
        },
        "comment": parsed["reasoning"],
    }

## 5.2 — Running the Pairwise Evaluation

Before executing a pairwise evaluation, two experiments must already exist. Both experiments must have been executed using **the same LangSmith dataset**, ensuring that every dataset example has a corresponding output from each experiment.

The `evaluate_comparative()` API compares the outputs of both experiments for every dataset example using the custom pairwise evaluator.

In [ ]:
pairwise_results = evaluate_comparative(
    (
        experiment_a.experiment_id,
        experiment_b.experiment_id,
    ),
    evaluators=[pairwise_preference],
    experiment_prefix="policy-bot-pairwise",
)

print(pairwise_results)

View the pairwise evaluation results at:
https://smith.langchain.com/o/2633fe0a-9587-4e9b-9ff1-685d7aaba413/datasets/2db03d05-cfe8-47ba-9958-6e7a20777f67/compare?selectedSessions=290130c2-04de-4a3a-86dd-a11f0cba4382%2Cff345445-183b-47aa-b1ea-19298d03b8ff&comparativeExperiment=5d194c97-84ab-4b98-a03e-59b5fcf929b1




  0%|          | 0/40 [00:00<?, ?it/s]

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 5.3 — Pairwise Evaluation Results

For each dataset example, LangSmith records:

- The output from **Experiment A**
- The output from **Experiment B**
- The preferred response (**A**, **B**, or **Tie**)
- The reasoning produced by the LLM evaluator

The resulting Pairwise Experiment enables side-by-side comparison of different prompt versions, models, or application implementations.

LangSmith also provides aggregate statistics showing how frequently each experiment was preferred across the entire dataset, making it easier to identify the stronger implementation.